# Notebook 04 — Baseline comparison on the same speaker folds

Run after Notebook 03. The CNN's primary evaluation is original-only. This notebook applies the same original-only validation and test policy to the classical baselines. Fold-specific normalization is fitted only on each fold's training partition.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

OUT = Path("anu_revision_output")
data = np.load(OUT / "speaker_split_mfcc_10fold_raw.npz", allow_pickle=True)

X_raw = data["X_train_raw"]
X_test_raw = data["X_test_raw"]
y = data["y_train"]
y_test_all = data["y_test"]
fold_id = data["fold_id"]
train_sources = data["train_original_ids"].astype(str)
test_is_original = data["test_is_original"].astype(bool)
classes = data["classes"].astype(str)

test_original_raw = X_test_raw[test_is_original]
y_test = y_test_all[test_is_original]

def norm_pair(X_fit_raw, X_eval_raw):
    mean = float(X_fit_raw.mean())
    std = float(X_fit_raw.std() + 1e-6)
    return ((X_fit_raw-mean)/std).reshape(len(X_fit_raw),-1), ((X_eval_raw-mean)/std).reshape(len(X_eval_raw),-1)

def weights_by_source(source_ids):
    _, inverse, counts = np.unique(source_ids, return_inverse=True, return_counts=True)
    w = 1.0 / counts[inverse]
    return (w * len(w) / w.sum()).astype(np.float32)

def metrics(y_true, y_pred):
    p,r,f,_=precision_recall_fscore_support(y_true,y_pred,average="macro",zero_division=0)
    return {"Accuracy":accuracy_score(y_true,y_pred),"Macro precision":p,"Macro recall":r,"Macro F1":f}

def make_models():
    return {
        "SVM (RBF)": SVC(kernel="rbf", class_weight="balanced"),
        "Random Forest": RandomForestClassifier(
            n_estimators=400, random_state=42, class_weight="balanced", n_jobs=-1
        ),
        "KNN (5-NN)": KNeighborsClassifier(n_neighbors=5)
    }

In [ ]:
# 10-fold CV: fit on augmented + original WAVs with per-source weights;
# calculate validation metrics on original WAVs only.
rows = []
for name in make_models():
    for fold in range(1, 11):
        fit_mask = fold_id != fold
        val_mask = (fold_id == fold) & data["train_is_original"].astype(bool)

        X_fit, X_val = norm_pair(X_raw[fit_mask], X_raw[val_mask])
        y_fit, y_val = y[fit_mask], y[val_mask]
        w = weights_by_source(train_sources[fit_mask])

        model = make_models()[name]
        if name in ["SVM (RBF)", "Random Forest"]:
            model.fit(X_fit, y_fit, sample_weight=w)
        else:
            # KNN does not support sample_weight; fit only original recordings.
            fit_original_mask = fit_mask & data["train_is_original"].astype(bool)
            X_fit_knn, X_val_knn = norm_pair(X_raw[fit_original_mask], X_raw[val_mask])
            model.fit(X_fit_knn, y[fit_original_mask])
            X_val = X_val_knn

        pred = model.predict(X_val)
        rows.append({"Model": name, "Fold": fold, **metrics(y_val, pred)})

cv_results = pd.DataFrame(rows)
display(cv_results)
cv_results.to_csv(OUT / "baseline_10fold_original_validation_results.csv", index=False)

cv_summary = cv_results.groupby("Model")[
    ["Accuracy","Macro precision","Macro recall","Macro F1"]
].agg(["mean","std"])
display(cv_summary)
cv_summary.to_csv(OUT / "baseline_10fold_original_validation_summary.csv")

In [ ]:
# Final baseline fit and original-only held-out test evaluation.
final_rows = []
for name, model in make_models().items():
    if name == "KNN (5-NN)":
        fit_mask = data["train_is_original"].astype(bool)
        X_fit, X_test = norm_pair(X_raw[fit_mask], test_original_raw)
        y_fit = y[fit_mask]
        model.fit(X_fit, y_fit)
    else:
        X_fit, X_test = norm_pair(X_raw, test_original_raw)
        w = weights_by_source(train_sources)
        model.fit(X_fit, y, sample_weight=w)

    pred = model.predict(X_test)
    final_rows.append({"Model":name, **metrics(y_test,pred)})
    joblib.dump(model, OUT / (name.lower().replace(" ","_").replace("(","").replace(")","") + "_original_test.joblib"))

comparison = pd.DataFrame(final_rows)
display(comparison)
comparison.to_csv(OUT / "baseline_10fold_original_final_test_comparison.csv", index=False)